Imports libraries

In [ ]:
#Chargement des données préprocessées
import pickle
import os
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

import matplotlib.pyplot as plt

# # Chargement du dataset généré par le notebook de preprocessing
df = pd.read_pickle(Path("../data/processed/dataset_final.pkl"))

Construction des représentations TF-IDF


In [ ]:
# TF-IDF 
#with open(Path("../data/embeddings/tfidf_vectorizer.pkl"), "rb") as f:
#   vectorizer = pickle.load(f)

# Texte nettoyé + tokenisé + lemmatisé 
texts = df["clean_tokens"].apply(lambda tokens: " ".join(tokens))


#TF-IDF sur texte déjà nettoyé
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.95
)

tfidf_emb = vectorizer.fit_transform(texts)


In [ ]:
# Entraînement du modèle LDA

lda_model = LatentDirichletAllocation(
    n_components=20,
    learning_method="online",
    random_state=42
)

lda_model.fit(tfidf_emb)


Affichage des résultats

In [ ]:
# Extraction des topics
# sauvegarde des top words par topic
def print_top_words(model, feature_names, n_top_words=20):
    topic_words = {}
    for topic_idx, topic in enumerate(model.components_):
        top_features = [
            feature_names[i]
            for i in topic.argsort()[:-n_top_words - 1:-1]
        ]
        print(f"Topic {topic_idx + 1}: {', '.join(top_features)}")
        topic_words[topic_idx] = top_features
    return topic_words

feature_names = vectorizer.get_feature_names_out()
topic_words = print_top_words(lda_model, feature_names)

#df_topic_words = pd.DataFrame(topic_words).T
#df_topic_words.to_csv("topic_words.csv", index=False)

In [ ]:
# Thème dominant par document
topics = np.argmax(lda_model.transform(tfidf_emb), axis=1)


Visualisation

In [ ]:
#Visualisation de la distribution des topics
import matplotlib.pyplot as plt

topics_size = pd.Series(topics).value_counts().sort_index()

plt.figure(figsize=(10, 6))
plt.bar(topics_size.index + 1, topics_size.values)
plt.xlabel("Topic")
plt.ylabel("Number of documents")
plt.title("LDA Topic Sizes")
plt.xticks(topics_size.index + 1)
plt.show()

In [ ]:
# Évaluation de la diversité des topics
def topic_diversity(topic_words):
    words = []
    for w in topic_words.values():
        words.extend(w)
    return len(set(words)) / len(words)

diversity_score = topic_diversity(topic_words)
print(f"Diversité des topics: {diversity_score:.2f}")


In [ ]:

clean_topics = []
for topic in topic_words.values():
    words = []
    for w in topic:
        words.extend(w.split())
    clean_topics.append(words)

tokenized_texts = df["clean_tokens"].tolist()

dictionary = Dictionary(tokenized_texts)

# Calcul de la cohérence des topics (c_v)
coherence_model = CoherenceModel(
    topics=clean_topics,
    texts=tokenized_texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score = coherence_model.get_coherence()
print("Coherence (c_v):", coherence_score)